In [ ]:
# ============================================================
# Section 7 - Minimal Isotonic Atlas and 4D/3D LUT Construction
# GitHub/reproducibility-ready single cell
#
# Notebook:
#   06_section7_lut_profile_evaluation.ipynb
#
# Purpose:
#   Build a condition-indexed voltage-SoC lookup-table atlas using
#   isotonic regression with light upper-tail regularization.
#
# Required input:
#   outputs/04_section5_statistical_analysis/tidy_clean.csv
#
# Optional input:
#   outputs/05_section6_calibration_evaluation/bandwise_metrics.csv
#
# Terminology:
#   Rate        = charge cut-off current level: C/5, C/40
#   Charge_Rate = discharge-rate/rate-history level: 0.7C, 1C, 2C
#
# Outputs:
#   outputs/06_section7_lut_profile_evaluation/lut_4d.csv
#   outputs/06_section7_lut_profile_evaluation/lut_3d_T10.csv
#   outputs/06_section7_lut_profile_evaluation/lut_3d_T25.csv
#   outputs/06_section7_lut_profile_evaluation/lut_3d_T45.csv
#   outputs/06_section7_lut_profile_evaluation/lut_3d_T60.csv
#   outputs/06_section7_lut_profile_evaluation/lut_fit_report.csv
#   outputs/06_section7_lut_profile_evaluation/lut_report.json
#   outputs/06_section7_lut_profile_evaluation/bandwise_delta_mae_mV.csv  [optional]
# ============================================================

import os
import json
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# Paths
# ============================================================

SECTION5_DIR = "outputs/04_section5_statistical_analysis"
SECTION6_DIR = "outputs/05_section6_calibration_evaluation"
OUT_DIR = "outputs/06_section7_lut_profile_evaluation"

os.makedirs(OUT_DIR, exist_ok=True)

TIDY_PATH = os.path.join(SECTION5_DIR, "tidy_clean.csv")
SECTION6_BANDWISE_METRICS_PATH = os.path.join(SECTION6_DIR, "bandwise_metrics.csv")

if not os.path.exists(TIDY_PATH):
    raise FileNotFoundError(
        "Required Section 5 input was not found. "
        "Run 04_section5_statistical_analysis.ipynb first.\n"
        f"Missing file: {TIDY_PATH}"
    )

print("Section 7 inputs:")
print(" - tidy_clean:", TIDY_PATH)
print(" - optional Section 6 bandwise metrics:", SECTION6_BANDWISE_METRICS_PATH)
print("Output directory:", OUT_DIR)


# ============================================================
# Column normalization
# ============================================================

def coerce_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def pick_exact_or_none(df, candidates):
    lower = {c.lower(): c for c in df.columns}

    for candidate in candidates:
        key = candidate.lower()
        if key in lower:
            return lower[key]

    return None


def pick_contains_or_none(df, pattern):
    pattern = pattern.lower()

    for col in df.columns:
        if pattern in col.lower():
            return col

    return None


def normalize_columns(df):
    """
    Normalize the Section 5 tidy table into the canonical LUT columns:

    - SoC
    - Voltage
    - Temperature
    - Rate
    - Charge_Rate

    Important:
    Rate is the charge cut-off current level.
    Charge_Rate is the discharge-rate/rate-history level.
    """
    df = df.copy()

    # SoC
    soc_col = pick_exact_or_none(
        df,
        ["SoC", "soc", "state_of_charge", "soc_%"]
    )

    if soc_col is None:
        soc_col = pick_contains_or_none(df, "soc")

    if soc_col is None:
        raise KeyError("SoC-like column was not found.")

    # Temperature
    temp_col = pick_exact_or_none(
        df,
        ["Temperature", "Temp", "ambient_temperature", "T"]
    )

    if temp_col is None:
        temp_col = pick_contains_or_none(df, "temp")

    if temp_col is None:
        raise KeyError("Temperature-like column was not found.")

    # Rate = charge cut-off current.
    # Do not accidentally pick Charge_Rate as Rate.
    rate_col = pick_exact_or_none(df, ["Rate"])

    if rate_col is None:
        rate_col = pick_exact_or_none(
            df,
            ["Cutoff", "Cut_off", "Charge_Cutoff", "Charge_Cut_Off"]
        )

    if rate_col is None:
        raise KeyError(
            "Rate column was not found. Expected internal label 'Rate' "
            "for charge cut-off current."
        )

    # Charge_Rate = discharge-rate/rate-history.
    charge_rate_col = pick_exact_or_none(
        df,
        ["Charge_Rate", "ChargeRate", "Charge-Rate", "CR"]
    )

    if charge_rate_col is None:
        raise KeyError(
            "Charge_Rate column was not found. Expected internal label "
            "'Charge_Rate' for discharge-rate/rate-history."
        )

    # Voltage
    voltage_source = None

    if "Voltage_clean" in df.columns:
        voltage_series = coerce_numeric(df["Voltage_clean"])
        voltage_source = "Voltage_clean"

    elif "Voltage" in df.columns:
        voltage_series = coerce_numeric(df["Voltage"])
        voltage_source = "Voltage"

    elif "V_median" in df.columns:
        voltage_series = coerce_numeric(df["V_median"])
        voltage_source = "V_median"

    elif "V_q25" in df.columns and "V_q75" in df.columns:
        voltage_series = (
            coerce_numeric(df["V_q25"]) + coerce_numeric(df["V_q75"])
        ) / 2.0
        voltage_source = "mean(V_q25,V_q75)"

    elif "VoltageV" in df.columns:
        voltage_series = coerce_numeric(df["VoltageV"])
        voltage_source = "VoltageV"

    else:
        alt_voltage_col = pick_contains_or_none(df, "volt")

        if alt_voltage_col is None:
            raise KeyError("Voltage-like column was not found.")

        voltage_series = coerce_numeric(df[alt_voltage_col])
        voltage_source = alt_voltage_col

    out = pd.DataFrame({
        "SoC": coerce_numeric(df[soc_col]),
        "Voltage": voltage_series,
        "Temperature": coerce_numeric(df[temp_col]),
        "Rate": df[rate_col].astype(str).str.strip(),
        "Charge_Rate": df[charge_rate_col].astype(str).str.strip(),
    })

    out = out.dropna(
        subset=["SoC", "Voltage", "Temperature", "Rate", "Charge_Rate"]
    ).copy()

    out["Temperature"] = out["Temperature"].astype(int)

    expected_temperature = [10, 25, 45, 60]
    expected_rate = ["C/5", "C/40"]
    expected_charge_rate = ["0.7C", "1C", "2C"]

    detected_temperature = [
        value for value in expected_temperature
        if value in sorted(out["Temperature"].unique().tolist())
    ]

    detected_rate = [
        value for value in expected_rate
        if value in sorted(out["Rate"].unique().tolist())
    ]

    detected_charge_rate = [
        value for value in expected_charge_rate
        if value in sorted(out["Charge_Rate"].unique().tolist())
    ]

    out = out[
        out["Temperature"].isin(detected_temperature)
        & out["Rate"].isin(detected_rate)
        & out["Charge_Rate"].isin(detected_charge_rate)
    ].copy()

    soc_bins = [0, 20, 60, 90, 100]
    soc_labels = ["0-20", "20-60", "60-90", "90-100"]

    out["SoC_band"] = pd.cut(
        out["SoC"].clip(0, 100),
        bins=soc_bins,
        labels=soc_labels,
        include_lowest=True,
    ).astype(str)

    print("\n[normalize]")
    print(f"Rows after cleaning: {len(out):,}")
    print(f"Voltage source: {voltage_source}")
    print(f"Temperature levels: {detected_temperature}")
    print(f"Rate / charge cut-off current levels: {detected_rate}")
    print(f"Charge_Rate / discharge-rate/rate-history levels: {detected_charge_rate}")

    return out, detected_temperature, detected_rate, detected_charge_rate, voltage_source


# ============================================================
# Load tidy data
# ============================================================

raw = pd.read_csv(TIDY_PATH)

tidy, TEMP_LEVELS, RATE_LEVELS, CHARGE_RATE_LEVELS, VOLTAGE_SOURCE = normalize_columns(raw)

if len(TEMP_LEVELS) == 4 and len(RATE_LEVELS) == 2 and len(CHARGE_RATE_LEVELS) == 3:
    print("\n[OK] Full 4 x 2 x 3 condition level set detected.")
else:
    print("\n[WARN] Full 4 x 2 x 3 level set was not detected.")
    print("       LUT will be generated only for detected levels.")


# ============================================================
# Isotonic PAV and upper-tail regularization
# ============================================================

def isotonic_pav(y, w=None):
    """
    Pool-Adjacent-Violators algorithm for a nondecreasing fit.

    This list-based implementation avoids block-index errors and does not
    require external dependencies.
    """
    y = np.asarray(y, dtype=float)
    n = len(y)

    if n == 0:
        return y.copy()

    if w is None:
        w = np.ones(n, dtype=float)
    else:
        w = np.asarray(w, dtype=float)

    values = list(y.astype(float))
    weights = list(w.astype(float))
    blocks = [[i] for i in range(n)]

    k = 0

    while k < len(values) - 1:
        if values[k] > values[k + 1]:
            new_weight = weights[k] + weights[k + 1]

            if new_weight <= 0:
                new_value = 0.5 * (values[k] + values[k + 1])
            else:
                new_value = (
                    weights[k] * values[k]
                    + weights[k + 1] * values[k + 1]
                ) / new_weight

            new_block = blocks[k] + blocks[k + 1]

            values.pop(k)
            values.pop(k)
            values.insert(k, new_value)

            weights.pop(k)
            weights.pop(k)
            weights.insert(k, new_weight)

            blocks.pop(k)
            blocks.pop(k)
            blocks.insert(k, new_block)

            if k > 0:
                k -= 1
        else:
            k += 1

    fitted = np.empty(n, dtype=float)

    for value, indices in zip(values, blocks):
        fitted[indices] = value

    return fitted


def aggregate_duplicate_soc(soc, voltage, weights):
    """
    Aggregate duplicate SoC entries before interpolation.

    This prevents ambiguous interpolation behavior when repeated SoC values
    are present in dense voltage-SoC traces.
    """
    df = pd.DataFrame({
        "SoC": soc,
        "Voltage": voltage,
        "weight": weights,
    })

    df = df.dropna(subset=["SoC", "Voltage", "weight"]).copy()

    if df.empty:
        return np.array([]), np.array([]), np.array([])

    grouped = (
        df.groupby("SoC", as_index=False)
        .apply(
            lambda g: pd.Series({
                "Voltage": np.average(g["Voltage"], weights=g["weight"]),
                "weight": float(g["weight"].sum()),
            })
        )
        .reset_index(drop=True)
        .sort_values("SoC")
    )

    return (
        grouped["SoC"].to_numpy(dtype=float),
        grouped["Voltage"].to_numpy(dtype=float),
        grouped["weight"].to_numpy(dtype=float),
    )


def enforce_convex_tail(x, y, tail_start_frac=0.90):
    """
    Light convex-tail enforcement over the upper SoC tail.

    The isotonic fit is already nondecreasing. This step only prevents
    decreasing incremental slopes in the upper tail.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(y) < 3:
        return y.copy()

    threshold = np.quantile(x, tail_start_frac)
    tail_idx = np.where(x >= threshold)[0]

    if tail_idx.size < 3:
        return y.copy()

    y_out = y.copy()
    diffs = np.diff(y_out[tail_idx])

    for i in range(1, len(diffs)):
        if diffs[i] < diffs[i - 1]:
            diffs[i] = diffs[i - 1]

    tail_values = [y_out[tail_idx[0]]]

    for diff in diffs:
        tail_values.append(tail_values[-1] + diff)

    y_out[tail_idx] = np.asarray(tail_values, dtype=float)

    return y_out


def fit_isotonic_curve(soc, voltage, top_frac=0.05, top_weight=0.2):
    """
    Fit a monotone voltage-SoC curve for one operating condition.
    """
    soc = np.asarray(soc, dtype=float)
    voltage = np.asarray(voltage, dtype=float)

    valid = np.isfinite(soc) & np.isfinite(voltage)

    soc = soc[valid]
    voltage = voltage[valid]

    if soc.size == 0:
        return np.array([]), np.array([])

    order = np.argsort(soc)
    soc = soc[order]
    voltage = voltage[order]

    weights = np.ones_like(voltage, dtype=float)

    top_threshold = 100.0 * (1.0 - float(top_frac))
    weights[soc >= top_threshold] = float(top_weight)

    soc_unique, voltage_unique, weights_unique = aggregate_duplicate_soc(
        soc,
        voltage,
        weights,
    )

    if soc_unique.size == 0:
        return np.array([]), np.array([])

    voltage_iso = isotonic_pav(voltage_unique, w=weights_unique)
    voltage_iso = enforce_convex_tail(
        soc_unique,
        voltage_iso,
        tail_start_frac=0.90,
    )

    return soc_unique, voltage_iso


def interpolate_to_grid(x, y, xgrid):
    """
    Interpolate a fitted curve to the deployment SoC grid.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    xgrid = np.asarray(xgrid, dtype=float)

    if x.size == 0:
        return np.full_like(xgrid, np.nan, dtype=float)

    return np.interp(xgrid, x, y, left=y[0], right=y[-1])


# ============================================================
# Fit LUT atlas
# ============================================================

SOC_GRID = np.arange(0.0, 101.0, 1.0)

lut_records = []
fit_report_records = []

for temperature in TEMP_LEVELS:
    for cutoff in RATE_LEVELS:
        for ratehist in CHARGE_RATE_LEVELS:

            group = tidy[
                (tidy["Temperature"] == temperature)
                & (tidy["Rate"] == cutoff)
                & (tidy["Charge_Rate"] == ratehist)
            ].copy()

            soc_fit, voltage_fit = fit_isotonic_curve(
                group["SoC"].to_numpy(dtype=float),
                group["Voltage"].to_numpy(dtype=float),
                top_frac=0.05,
                top_weight=0.2,
            )

            voltage_grid = interpolate_to_grid(
                soc_fit,
                voltage_fit,
                SOC_GRID,
            )

            fit_report_records.append({
                "Temperature": temperature,
                "Rate_charge_cutoff": cutoff,
                "Charge_Rate_ratehist": ratehist,
                "n_raw_points": int(len(group)),
                "n_fit_points": int(len(soc_fit)),
                "v_grid_nan_count": int(np.isnan(voltage_grid).sum()),
                "soc_min_fit": float(np.nanmin(soc_fit)) if len(soc_fit) else np.nan,
                "soc_max_fit": float(np.nanmax(soc_fit)) if len(soc_fit) else np.nan,
                "v_min_lut": float(np.nanmin(voltage_grid)) if np.isfinite(voltage_grid).any() else np.nan,
                "v_max_lut": float(np.nanmax(voltage_grid)) if np.isfinite(voltage_grid).any() else np.nan,
            })

            for soc_value, v_lut in zip(SOC_GRID, voltage_grid):
                lut_records.append({
                    "Temperature": temperature,
                    "Rate": cutoff,
                    "Charge_Rate": ratehist,
                    "SoC": float(soc_value),
                    "V_lut": float(v_lut) if np.isfinite(v_lut) else np.nan,
                })

lut4d = pd.DataFrame.from_records(lut_records)
fit_report = pd.DataFrame.from_records(fit_report_records)


# ============================================================
# Save 4D and per-temperature 3D LUTs
# ============================================================

lut4d_path = os.path.join(OUT_DIR, "lut_4d.csv")
fit_report_path = os.path.join(OUT_DIR, "lut_fit_report.csv")

lut4d.to_csv(lut4d_path, index=False)
fit_report.to_csv(fit_report_path, index=False)

per_temperature_paths = []

for temperature in TEMP_LEVELS:
    sub = lut4d[
        lut4d["Temperature"] == temperature
    ][["SoC", "Rate", "Charge_Rate", "V_lut"]].copy()

    path = os.path.join(OUT_DIR, f"lut_3d_T{temperature}.csv")
    sub.to_csv(path, index=False)
    per_temperature_paths.append(path)


# ============================================================
# LUT specification and memory report
# ============================================================

n_entries = int(lut4d.shape[0])
expected_full_entries = 4 * 2 * 3 * len(SOC_GRID)

bytes_f32 = n_entries * 4
bytes_f16 = n_entries * 2

per_temperature_entries = int(len(RATE_LEVELS) * len(CHARGE_RATE_LEVELS) * len(SOC_GRID))
per_temperature_bytes_f32 = per_temperature_entries * 4
per_temperature_bytes_f16 = per_temperature_entries * 2

lut_specification = pd.DataFrame([{
    "Temperature_levels": ",".join(map(str, TEMP_LEVELS)),
    "Rate_charge_cutoff_levels": ",".join(RATE_LEVELS),
    "Charge_Rate_ratehist_levels": ",".join(CHARGE_RATE_LEVELS),
    "SoC_grid_start": float(SOC_GRID[0]),
    "SoC_grid_stop": float(SOC_GRID[-1]),
    "SoC_grid_step": 1.0,
    "SoC_grid_n_points": int(len(SOC_GRID)),
    "total_entries": n_entries,
    "expected_full_4x2x3_entries": expected_full_entries,
    "float32_kB": round(bytes_f32 / 1024.0, 2),
    "float16_kB": round(bytes_f16 / 1024.0, 2),
}])

lut_specification_path = os.path.join(OUT_DIR, "lut_specification.csv")
lut_specification.to_csv(lut_specification_path, index=False)

lut_report = {
    "method": {
        "name": "minimal isotonic voltage-SoC LUT atlas",
        "isotonic_direction": "nondecreasing voltage as SoC increases",
        "top_tail_weighting": {
            "top_frac": 0.05,
            "top_weight": 0.2,
        },
        "convex_tail": {
            "enabled": True,
            "tail_start_quantile": 0.90,
        },
        "duplicate_soc_handling": "weighted aggregation before isotonic fitting",
    },
    "terminology": {
        "Rate": "charge cut-off current level",
        "Charge_Rate": "discharge-rate/rate-history level",
    },
    "inputs": {
        "tidy_clean": TIDY_PATH,
        "voltage_source": VOLTAGE_SOURCE,
    },
    "levels": {
        "Temperature": TEMP_LEVELS,
        "Rate_charge_cutoff": RATE_LEVELS,
        "Charge_Rate_ratehist": CHARGE_RATE_LEVELS,
    },
    "soc_grid": {
        "start": float(SOC_GRID[0]),
        "stop": float(SOC_GRID[-1]),
        "step": 1.0,
        "n_points": int(len(SOC_GRID)),
    },
    "entries": {
        "total_entries": n_entries,
        "expected_full_4x2x3_entries": expected_full_entries,
        "per_temperature_entries": per_temperature_entries,
    },
    "memory": {
        "float32_bytes": bytes_f32,
        "float32_kB": round(bytes_f32 / 1024.0, 2),
        "float16_bytes": bytes_f16,
        "float16_kB": round(bytes_f16 / 1024.0, 2),
        "per_temperature_float32_kB": round(per_temperature_bytes_f32 / 1024.0, 2),
        "per_temperature_float16_kB": round(per_temperature_bytes_f16 / 1024.0, 2),
    },
    "files": {
        "lut_4d_csv": lut4d_path,
        "lut_3d_by_temperature": per_temperature_paths,
        "fit_report": fit_report_path,
        "lut_specification": lut_specification_path,
    },
}

lut_report_path = os.path.join(OUT_DIR, "lut_report.json")

with open(lut_report_path, "w", encoding="utf-8") as f:
    json.dump(lut_report, f, indent=2, ensure_ascii=False)


# ============================================================
# Optional Section 6 band-wise Delta MAE mini-table
# ============================================================

def try_export_bandwise_delta_mae():
    """
    Optional helper:
    If Section 6 bandwise_metrics.csv exists, export a compact long-format
    Delta MAE table into the Section 7 output directory.
    """
    if not os.path.exists(SECTION6_BANDWISE_METRICS_PATH):
        print("\n[INFO] Section 6 bandwise_metrics.csv not found; skipping Delta MAE mini-table.")
        return None

    bandwise_metrics = pd.read_csv(SECTION6_BANDWISE_METRICS_PATH)

    required = {"SoC_band", "Variant", "MAE"}

    if not required.issubset(bandwise_metrics.columns):
        print("\n[WARN] bandwise_metrics.csv missing required columns; skipping Delta MAE mini-table.")
        return None

    bandwise_metrics["Variant"] = bandwise_metrics["Variant"].astype(str).str.strip()
    bandwise_metrics["Variant"] = bandwise_metrics["Variant"].replace({
        "Charge": "RateHist",
        "Charge-only": "RateHist",
        "RateHist-only": "RateHist",
        "Thermal-only": "Thermal",
    })

    baseline = (
        bandwise_metrics[
            bandwise_metrics["Variant"].str.lower() == "baseline"
        ][["SoC_band", "MAE"]]
        .rename(columns={"MAE": "MAE_base"})
    )

    if baseline.empty:
        print("\n[WARN] Baseline not found in bandwise_metrics.csv; skipping Delta MAE mini-table.")
        return None

    rows = []

    for variant in ["Thermal", "RateHist", "Both"]:
        variant_df = (
            bandwise_metrics[
                bandwise_metrics["Variant"].str.lower() == variant.lower()
            ][["SoC_band", "MAE"]]
            .rename(columns={"MAE": "MAE_variant"})
        )

        if variant_df.empty:
            print(f"[WARN] Variant not found in bandwise_metrics.csv: {variant}")
            continue

        merged = pd.merge(baseline, variant_df, on="SoC_band", how="inner")
        merged["Variant"] = variant
        merged["Delta_MAE_mV"] = (
            merged["MAE_base"] - merged["MAE_variant"]
        ) * 1000.0

        rows.append(
            merged[["SoC_band", "Variant", "Delta_MAE_mV"]]
        )

    if not rows:
        print("\n[WARN] No variants available for Delta MAE mini-table.")
        return None

    out = pd.concat(rows, ignore_index=True)

    out_path = os.path.join(OUT_DIR, "bandwise_delta_mae_mV.csv")
    out.to_csv(out_path, index=False)

    print("\n=== Optional band-wise Delta MAE (mV) vs Baseline ===")
    display(
        out.pivot(
            index="SoC_band",
            columns="Variant",
            values="Delta_MAE_mV"
        ).round(3)
    )
    print("Saved:", out_path)

    return out_path


optional_delta_mae_path = try_export_bandwise_delta_mae()


# ============================================================
# Run summary
# ============================================================

run_summary = {
    "outputs": {
        "lut_4d": lut4d_path,
        "lut_3d_by_temperature": per_temperature_paths,
        "lut_fit_report": fit_report_path,
        "lut_report": lut_report_path,
        "lut_specification": lut_specification_path,
        "optional_bandwise_delta_mae": optional_delta_mae_path,
    },
    "detected_levels": {
        "Temperature": TEMP_LEVELS,
        "Rate_charge_cutoff": RATE_LEVELS,
        "Charge_Rate_ratehist": CHARGE_RATE_LEVELS,
    },
    "n_lut_entries": n_entries,
    "expected_full_4x2x3_entries": expected_full_entries,
}

run_summary_path = os.path.join(OUT_DIR, "sec7_run_summary.json")

with open(run_summary_path, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2, ensure_ascii=False)


print("\n=== Section 7 - Isotonic Atlas and LUTs ===")
print("Output directory:", OUT_DIR)
print("Temperature levels:", TEMP_LEVELS)
print("Rate / charge cut-off current levels:", RATE_LEVELS)
print("Charge_Rate / discharge-rate/rate-history levels:", CHARGE_RATE_LEVELS)
print("SoC grid points:", len(SOC_GRID))
print("Total LUT entries:", n_entries)
print("Expected full 4 x 2 x 3 entries:", expected_full_entries)
print("Approximate memory float32:", round(bytes_f32 / 1024.0, 2), "kB")
print("Approximate memory float16:", round(bytes_f16 / 1024.0, 2), "kB")
print("Per-temperature sub-LUT float32:", round(per_temperature_bytes_f32 / 1024.0, 2), "kB")
print("Per-temperature sub-LUT float16:", round(per_temperature_bytes_f16 / 1024.0, 2), "kB")

print("\nSaved files:")
print(" -", lut4d_path)

for path in per_temperature_paths:
    print(" -", path)

print(" -", fit_report_path)
print(" -", lut_specification_path)
print(" -", lut_report_path)
print(" -", run_summary_path)

print("\n--- lut_4d head ---")
display(lut4d.head(10))

print("\n--- LUT fit report ---")
display(fit_report)

print("\n--- LUT specification ---")
display(lut_specification)

In [ ]:
# ============================================================
# Section 7 - Measured-grid LUT prediction and evaluation
# GitHub/reproducibility-ready single cell
#
# Notebook:
#   06_section7_lut_profile_evaluation.ipynb
#
# Required previous cell:
#   Section 7 isotonic LUT atlas construction
#
# Inputs:
#   outputs/04_section5_statistical_analysis/tidy_clean.csv
#   outputs/06_section7_lut_profile_evaluation/lut_4d.csv
#
# Outputs:
#   outputs/06_section7_lut_profile_evaluation/pred.csv
#   outputs/06_section7_lut_profile_evaluation/pred_run_report.json
#   outputs/06_section7_lut_profile_evaluation/overall_metrics_from_pred.csv
#   outputs/06_section7_lut_profile_evaluation/improvement_from_pred.csv
#   outputs/06_section7_lut_profile_evaluation/bandwise_metrics_from_pred.csv
#
# Variants:
#   Pred_Baseline : anchor LUT at T=25, Rate=C/40, Charge_Rate=1C
#   Pred_Thermal  : temperature-indexed LUT at row T, Rate=C/40, Charge_Rate=1C
#   Pred_TR       : full measured-grid LUT at row T, row Rate, row Charge_Rate
#
# Important:
#   No nearest-temperature fallback.
#   No silent extrapolation to missing operating tuples.
#
# Terminology:
#   Rate        = charge cut-off current level: C/5, C/40
#   Charge_Rate = discharge-rate/rate-history level: 0.7C, 1C, 2C
# ============================================================

import os
import json
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# Paths
# ============================================================

SECTION5_DIR = "outputs/04_section5_statistical_analysis"
OUT_DIR = "outputs/06_section7_lut_profile_evaluation"

os.makedirs(OUT_DIR, exist_ok=True)

TIDY_PATH = os.path.join(SECTION5_DIR, "tidy_clean.csv")
LUT_PATH = os.path.join(OUT_DIR, "lut_4d.csv")

if not os.path.exists(TIDY_PATH):
    raise FileNotFoundError(
        "Required Section 5 tidy file was not found. "
        "Run 04_section5_statistical_analysis.ipynb first.\n"
        f"Missing file: {TIDY_PATH}"
    )

if not os.path.exists(LUT_PATH):
    raise FileNotFoundError(
        "Required Section 7 LUT file was not found. "
        "Run the Section 7 isotonic LUT atlas cell first.\n"
        f"Missing file: {LUT_PATH}"
    )

print("Section 7 prediction inputs:")
print(" - tidy_clean:", TIDY_PATH)
print(" - lut_4d    :", LUT_PATH)
print("Output directory:", OUT_DIR)


# ============================================================
# Load data
# ============================================================

tidy = pd.read_csv(TIDY_PATH)
lut = pd.read_csv(LUT_PATH)


# ============================================================
# Validation and normalization
# ============================================================

required_tidy = {"SoC", "Temperature", "Rate", "Charge_Rate"}
missing_tidy = required_tidy - set(tidy.columns)

if missing_tidy:
    raise KeyError(f"tidy_clean.csv is missing required columns: {missing_tidy}")

if "Voltage_clean" in tidy.columns:
    vcol = "Voltage_clean"
elif "Voltage" in tidy.columns:
    vcol = "Voltage"
elif "VoltageV" in tidy.columns:
    vcol = "VoltageV"
elif "V_median" in tidy.columns:
    tidy["Voltage"] = pd.to_numeric(tidy["V_median"], errors="coerce")
    vcol = "Voltage"
elif "V_q25" in tidy.columns and "V_q75" in tidy.columns:
    tidy["Voltage"] = (
        pd.to_numeric(tidy["V_q25"], errors="coerce")
        + pd.to_numeric(tidy["V_q75"], errors="coerce")
    ) / 2.0
    vcol = "Voltage"
else:
    raise KeyError(
        "True voltage column was not found. Expected one of: "
        "Voltage_clean, Voltage, VoltageV, V_median, or V_q25/V_q75."
    )

required_lut = {"Temperature", "Rate", "Charge_Rate", "SoC", "V_lut"}
missing_lut = required_lut - set(lut.columns)

if missing_lut:
    raise KeyError(f"lut_4d.csv is missing required columns: {missing_lut}")

# Normalize tidy table.
tidy = tidy.copy()

tidy["SoC"] = pd.to_numeric(tidy["SoC"], errors="coerce")
tidy["Temperature"] = pd.to_numeric(tidy["Temperature"], errors="coerce")
tidy[vcol] = pd.to_numeric(tidy[vcol], errors="coerce")

tidy["Rate"] = tidy["Rate"].astype(str).str.strip()
tidy["Charge_Rate"] = tidy["Charge_Rate"].astype(str).str.strip()

tidy = tidy.dropna(
    subset=["SoC", "Temperature", vcol, "Rate", "Charge_Rate"]
).copy()

tidy["Temperature"] = tidy["Temperature"].astype(int)

# Normalize LUT table.
lut = lut.copy()

lut["SoC"] = pd.to_numeric(lut["SoC"], errors="coerce")
lut["Temperature"] = pd.to_numeric(lut["Temperature"], errors="coerce")
lut["V_lut"] = pd.to_numeric(lut["V_lut"], errors="coerce")

lut["Rate"] = lut["Rate"].astype(str).str.strip()
lut["Charge_Rate"] = lut["Charge_Rate"].astype(str).str.strip()

lut = lut.dropna(
    subset=["SoC", "Temperature", "V_lut", "Rate", "Charge_Rate"]
).copy()

lut["Temperature"] = lut["Temperature"].astype(int)

# Expected factorial levels.
expected_temperature = [10, 25, 45, 60]
expected_rate = ["C/5", "C/40"]
expected_charge_rate = ["0.7C", "1C", "2C"]

tidy = tidy[
    tidy["Temperature"].isin(expected_temperature)
    & tidy["Rate"].isin(expected_rate)
    & tidy["Charge_Rate"].isin(expected_charge_rate)
].copy()

lut = lut[
    lut["Temperature"].isin(expected_temperature)
    & lut["Rate"].isin(expected_rate)
    & lut["Charge_Rate"].isin(expected_charge_rate)
].copy()

if tidy.empty:
    raise RuntimeError("tidy_clean table is empty after expected-level filtering.")

if lut.empty:
    raise RuntimeError("LUT table is empty after expected-level filtering.")

print("\n[OK] Normalized inputs")
print(f"tidy rows: {len(tidy):,}")
print(f"LUT rows : {len(lut):,}")
print(f"Voltage true column: {vcol}")


# ============================================================
# LUT lookup without fallback
# ============================================================

def get_lut_curve(df_lut, temperature, rate, charge_rate):
    """
    Return one LUT curve for an exact measured-grid tuple.

    No nearest-condition fallback is allowed.
    """
    sub = df_lut[
        (df_lut["Temperature"] == int(temperature))
        & (df_lut["Rate"] == str(rate))
        & (df_lut["Charge_Rate"] == str(charge_rate))
    ].copy()

    if sub.empty:
        raise KeyError(
            "No LUT entry for measured-grid tuple: "
            f"Temperature={temperature}, Rate={rate}, Charge_Rate={charge_rate}. "
            "No nearest-condition fallback is allowed."
        )

    sub = sub.sort_values("SoC")

    xs = sub["SoC"].to_numpy(dtype=float)
    ys = sub["V_lut"].to_numpy(dtype=float)

    if xs.size == 0 or ys.size == 0:
        raise RuntimeError(
            "Empty LUT curve after filtering: "
            f"Temperature={temperature}, Rate={rate}, Charge_Rate={charge_rate}."
        )

    if np.isnan(ys).any():
        raise RuntimeError(
            "NaN values found in LUT curve: "
            f"Temperature={temperature}, Rate={rate}, Charge_Rate={charge_rate}."
        )

    return xs, ys


def lut_predict(df_lut, temperature, rate, charge_rate, soc_values):
    """
    Interpolate the exact LUT curve at the requested SoC values.
    """
    xs, ys = get_lut_curve(
        df_lut,
        temperature=temperature,
        rate=rate,
        charge_rate=charge_rate,
    )

    return np.interp(np.asarray(soc_values, dtype=float), xs, ys)


# ============================================================
# Pre-check all required measured-grid tuples
# ============================================================

required_full_tuples = (
    tidy[["Temperature", "Rate", "Charge_Rate"]]
    .drop_duplicates()
    .sort_values(["Temperature", "Rate", "Charge_Rate"])
)

lut_tuples = set(
    map(
        tuple,
        lut[["Temperature", "Rate", "Charge_Rate"]]
        .drop_duplicates()
        .to_numpy()
    )
)

missing_tuples = []

for _, row in required_full_tuples.iterrows():
    tup = (
        int(row["Temperature"]),
        str(row["Rate"]),
        str(row["Charge_Rate"]),
    )

    if tup not in lut_tuples:
        missing_tuples.append(tup)

if missing_tuples:
    raise KeyError(
        "Missing LUT tuples required by tidy evaluation rows. "
        f"Missing examples: {missing_tuples[:10]}"
    )

# Baseline anchor check.
_ = get_lut_curve(
    lut,
    temperature=25,
    rate="C/40",
    charge_rate="1C",
)

# Thermal anchor checks:
# row Temperature, fixed Rate=C/40, fixed Charge_Rate=1C.
for temperature in sorted(tidy["Temperature"].unique()):
    _ = get_lut_curve(
        lut,
        temperature=int(temperature),
        rate="C/40",
        charge_rate="1C",
    )

print("\n[OK] All required measured-grid LUT tuples are present.")
print("[OK] Baseline and thermal anchor tuples are present.")


# ============================================================
# Build prediction table
# ============================================================

base_columns = ["SoC", "Temperature", "Rate", "Charge_Rate", vcol]
extra_columns = []

for col in ["SoC_band", "sample_number", "sample_int", "Cycle_Index", "Step_Index"]:
    if col in tidy.columns:
        extra_columns.append(col)

pred = tidy[extra_columns + base_columns].copy()
pred = pred.rename(columns={vcol: "Voltage_true"})

# Baseline prediction:
# fixed anchor LUT for all rows.
pred["Pred_Baseline"] = lut_predict(
    lut,
    temperature=25,
    rate="C/40",
    charge_rate="1C",
    soc_values=pred["SoC"].to_numpy(dtype=float),
)

# Thermal prediction:
# row temperature, fixed Rate=C/40 and Charge_Rate=1C.
pred["Pred_Thermal"] = np.nan

for temperature, idx in pred.groupby("Temperature").groups.items():
    soc_values = pred.loc[idx, "SoC"].to_numpy(dtype=float)

    pred.loc[idx, "Pred_Thermal"] = lut_predict(
        lut,
        temperature=int(temperature),
        rate="C/40",
        charge_rate="1C",
        soc_values=soc_values,
    )

# Full measured-grid prediction:
# row temperature, row Rate, row Charge_Rate.
pred["Pred_TR"] = np.nan

for (temperature, rate, charge_rate), idx in pred.groupby(
    ["Temperature", "Rate", "Charge_Rate"]
).groups.items():
    soc_values = pred.loc[idx, "SoC"].to_numpy(dtype=float)

    pred.loc[idx, "Pred_TR"] = lut_predict(
        lut,
        temperature=int(temperature),
        rate=str(rate),
        charge_rate=str(charge_rate),
        soc_values=soc_values,
    )


# ============================================================
# Residuals and absolute errors
# ============================================================

pred["Res_Baseline"] = pred["Voltage_true"] - pred["Pred_Baseline"]
pred["Res_Thermal"] = pred["Voltage_true"] - pred["Pred_Thermal"]
pred["Res_TR"] = pred["Voltage_true"] - pred["Pred_TR"]

pred["AbsErr_Baseline"] = pred["Res_Baseline"].abs()
pred["AbsErr_Thermal"] = pred["Res_Thermal"].abs()
pred["AbsErr_TR"] = pred["Res_TR"].abs()

if "SoC_band" not in pred.columns:
    pred["SoC_band"] = pd.cut(
        pred["SoC"].clip(0, 100),
        bins=[0, 20, 60, 90, 100],
        labels=["0-20", "20-60", "60-90", "90-100"],
        include_lowest=True,
    ).astype(str)


# ============================================================
# Metrics
# ============================================================

def rmse(values):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]

    if arr.size == 0:
        return np.nan

    return float(np.sqrt(np.mean(arr ** 2)))


def mae(values):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]

    if arr.size == 0:
        return np.nan

    return float(np.mean(np.abs(arr)))


overall = pd.DataFrame([
    {
        "Variant": "Baseline",
        "MAE": mae(pred["Res_Baseline"]),
        "RMSE": rmse(pred["Res_Baseline"]),
        "N": int(pred["Res_Baseline"].notna().sum()),
    },
    {
        "Variant": "Thermal",
        "MAE": mae(pred["Res_Thermal"]),
        "RMSE": rmse(pred["Res_Thermal"]),
        "N": int(pred["Res_Thermal"].notna().sum()),
    },
    {
        "Variant": "T+Rate",
        "MAE": mae(pred["Res_TR"]),
        "RMSE": rmse(pred["Res_TR"]),
        "N": int(pred["Res_TR"].notna().sum()),
    },
])

baseline_mae = float(overall.loc[overall["Variant"] == "Baseline", "MAE"].iloc[0])
baseline_rmse = float(overall.loc[overall["Variant"] == "Baseline", "RMSE"].iloc[0])

improvement = overall.copy()
improvement["Delta_MAE_V"] = baseline_mae - improvement["MAE"]
improvement["Delta_RMSE_V"] = baseline_rmse - improvement["RMSE"]
improvement["Delta_MAE_mV"] = 1000.0 * improvement["Delta_MAE_V"]
improvement["Delta_RMSE_mV"] = 1000.0 * improvement["Delta_RMSE_V"]
improvement["MAE_reduction_pct"] = (
    100.0 * improvement["Delta_MAE_V"] / baseline_mae
    if baseline_mae > 0 else np.nan
)
improvement["RMSE_reduction_pct"] = (
    100.0 * improvement["Delta_RMSE_V"] / baseline_rmse
    if baseline_rmse > 0 else np.nan
)

bandwise_rows = []

for band, group in pred.groupby("SoC_band"):
    bandwise_rows.append({
        "SoC_band": band,
        "Variant": "Baseline",
        "MAE": mae(group["Res_Baseline"]),
        "RMSE": rmse(group["Res_Baseline"]),
        "N": int(group["Res_Baseline"].notna().sum()),
    })
    bandwise_rows.append({
        "SoC_band": band,
        "Variant": "Thermal",
        "MAE": mae(group["Res_Thermal"]),
        "RMSE": rmse(group["Res_Thermal"]),
        "N": int(group["Res_Thermal"].notna().sum()),
    })
    bandwise_rows.append({
        "SoC_band": band,
        "Variant": "T+Rate",
        "MAE": mae(group["Res_TR"]),
        "RMSE": rmse(group["Res_TR"]),
        "N": int(group["Res_TR"].notna().sum()),
    })

bandwise = pd.DataFrame(bandwise_rows)


# ============================================================
# Save outputs
# ============================================================

pred_path = os.path.join(OUT_DIR, "pred.csv")
overall_path = os.path.join(OUT_DIR, "overall_metrics_from_pred.csv")
improvement_path = os.path.join(OUT_DIR, "improvement_from_pred.csv")
bandwise_path = os.path.join(OUT_DIR, "bandwise_metrics_from_pred.csv")
report_path = os.path.join(OUT_DIR, "pred_run_report.json")

pred.to_csv(pred_path, index=False)
overall.to_csv(overall_path, index=False)
improvement.to_csv(improvement_path, index=False)
bandwise.to_csv(bandwise_path, index=False)

run_report = {
    "inputs": {
        "tidy_clean": TIDY_PATH,
        "lut_4d": LUT_PATH,
    },
    "outputs": {
        "pred": pred_path,
        "overall_metrics_from_pred": overall_path,
        "improvement_from_pred": improvement_path,
        "bandwise_metrics_from_pred": bandwise_path,
    },
    "n_rows": int(len(pred)),
    "voltage_true_column": "Voltage_true",
    "source_voltage_column": vcol,
    "variants": {
        "Baseline": "T=25, Rate=C/40, Charge_Rate=1C anchor LUT",
        "Thermal": "row Temperature, Rate=C/40, Charge_Rate=1C",
        "T+Rate": "row Temperature, row Rate, row Charge_Rate full measured-grid LUT",
    },
    "terminology": {
        "Rate": "charge cut-off current level",
        "Charge_Rate": "discharge-rate/rate-history level",
    },
    "no_fallback_policy": True,
    "required_tuples_checked": int(len(required_full_tuples)),
    "missing_tuples": [],
}

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(run_report, f, indent=2, ensure_ascii=False)


# ============================================================
# Print summary
# ============================================================

print("\nSaved predictions to:", pred_path)
print("Saved run report to:", report_path)
print("Saved quick metrics to:", overall_path)
print("Saved improvement metrics to:", improvement_path)
print("Saved bandwise metrics to:", bandwise_path)

print("\n=== Section 7 quick metrics from pred.csv ===")
display(overall)

print("\n=== Section 7 improvement vs baseline ===")
display(improvement)

print("\n=== Section 7 bandwise metrics ===")
display(bandwise)

print("\nPreview of pred.csv:")
display(pred.head(8))

In [ ]:
# ============================================================
# Section 7 - Bias, parity, residual diagnostics, and metrics
# GitHub/reproducibility-ready single cell
#
# Notebook:
#   06_section7_lut_profile_evaluation.ipynb
#
# Required previous cell:
#   Section 7 measured-grid LUT prediction and evaluation
#
# Input:
#   outputs/06_section7_lut_profile_evaluation/pred.csv
#
# Expected true-voltage column:
#   Voltage_true
#
# Required prediction columns:
#   Pred_Baseline
#   Pred_Thermal
#   Pred_TR
#
# Outputs:
#   outputs/06_section7_lut_profile_evaluation/overall_metrics_from_pred.csv
#   outputs/06_section7_lut_profile_evaluation/sec7_ablation_vs_baseline.csv
#   outputs/06_section7_lut_profile_evaluation/sec7_bandwise_delta_mae_mV.csv
#   outputs/06_section7_lut_profile_evaluation/sec7_parity_regression.csv
#   outputs/06_section7_lut_profile_evaluation/fig_bias_mean_residual.png
#   outputs/06_section7_lut_profile_evaluation/fig_parity_triptych.png
#   outputs/06_section7_lut_profile_evaluation/fig_residual_vs_soc.png
#   outputs/06_section7_lut_profile_evaluation/sec7_diagnostics_run_summary.json
#
# Variants:
#   Baseline : anchor LUT at T=25, Rate=C/40, Charge_Rate=1C
#   Thermal  : temperature-indexed LUT at row T, Rate=C/40, Charge_Rate=1C
#   T+Rate   : full measured-grid LUT at row T, row Rate, row Charge_Rate
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# Paths
# ============================================================

OUT_DIR = "outputs/06_section7_lut_profile_evaluation"
os.makedirs(OUT_DIR, exist_ok=True)

PRED_PATH = os.path.join(OUT_DIR, "pred.csv")

if not os.path.exists(PRED_PATH):
    raise FileNotFoundError(
        "pred.csv was not found. Run the Section 7 measured-grid "
        "prediction cell first.\n"
        f"Missing file: {PRED_PATH}"
    )

print("Input pred:", PRED_PATH)
print("Output directory:", OUT_DIR)


# ============================================================
# Load pred.csv
# ============================================================

df = pd.read_csv(PRED_PATH)


# ============================================================
# Validate required columns
# ============================================================

if "Voltage_true" in df.columns:
    ycol = "Voltage_true"
elif "Voltage_clean" in df.columns:
    ycol = "Voltage_clean"
elif "Voltage" in df.columns:
    ycol = "Voltage"
elif "VoltageV" in df.columns:
    ycol = "VoltageV"
else:
    raise KeyError(
        "True voltage column was not found. Expected one of: "
        "Voltage_true, Voltage_clean, Voltage, or VoltageV."
    )

required_columns = {
    ycol,
    "SoC",
    "Pred_Baseline",
    "Pred_Thermal",
    "Pred_TR",
}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise KeyError(f"Missing required columns in pred.csv: {missing_columns}")

for col in [ycol, "SoC", "Pred_Baseline", "Pred_Thermal", "Pred_TR"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(
    subset=[ycol, "SoC", "Pred_Baseline", "Pred_Thermal", "Pred_TR"]
).copy()

if df.empty:
    raise RuntimeError("pred.csv has no usable rows after numeric cleanup.")

print("\n[OK] Prediction table loaded")
print(f"Rows used: {len(df):,}")
print(f"True-voltage column: {ycol}")


# ============================================================
# Residuals and absolute errors
# ============================================================

df["Res_Baseline"] = df[ycol] - df["Pred_Baseline"]
df["Res_Thermal"] = df[ycol] - df["Pred_Thermal"]
df["Res_TR"] = df[ycol] - df["Pred_TR"]

df["AbsErr_Baseline"] = df["Res_Baseline"].abs()
df["AbsErr_Thermal"] = df["Res_Thermal"].abs()
df["AbsErr_TR"] = df["Res_TR"].abs()


# ============================================================
# SoC bands
# ============================================================

SOC_BINS = [0, 20, 60, 90, 100]
SOC_LABELS = ["0-20", "20-60", "60-90", "90-100"]

if "SoC_band" in df.columns:
    df["SoC_band"] = df["SoC_band"].astype(str)
else:
    df["SoC_band"] = pd.cut(
        df["SoC"].clip(0, 100),
        bins=SOC_BINS,
        labels=SOC_LABELS,
        include_lowest=True,
    ).astype(str)

df["SoC_band"] = df["SoC_band"].replace({
    "0.0-20.0": "0-20",
    "20.0-60.0": "20-60",
    "60.0-90.0": "60-90",
    "90.0-100.0": "90-100",
})


# ============================================================
# Metric helpers
# ============================================================

def mae_from_residual(residual):
    arr = np.asarray(residual, dtype=float)
    arr = arr[np.isfinite(arr)]

    if arr.size == 0:
        return np.nan

    return float(np.mean(np.abs(arr)))


def rmse_from_residual(residual):
    arr = np.asarray(residual, dtype=float)
    arr = arr[np.isfinite(arr)]

    if arr.size == 0:
        return np.nan

    return float(np.sqrt(np.mean(arr ** 2)))


def save_current_figure(fig_name):
    path = os.path.join(OUT_DIR, fig_name)
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved:", path)
    return path


# ============================================================
# Overall metrics
# ============================================================

overall = pd.DataFrame([
    {
        "Variant": "Baseline",
        "MAE": mae_from_residual(df["Res_Baseline"]),
        "RMSE": rmse_from_residual(df["Res_Baseline"]),
        "N": int(df["Res_Baseline"].notna().sum()),
    },
    {
        "Variant": "Thermal",
        "MAE": mae_from_residual(df["Res_Thermal"]),
        "RMSE": rmse_from_residual(df["Res_Thermal"]),
        "N": int(df["Res_Thermal"].notna().sum()),
    },
    {
        "Variant": "T+Rate",
        "MAE": mae_from_residual(df["Res_TR"]),
        "RMSE": rmse_from_residual(df["Res_TR"]),
        "N": int(df["Res_TR"].notna().sum()),
    },
])

overall_path = os.path.join(OUT_DIR, "overall_metrics_from_pred.csv")
overall.to_csv(overall_path, index=False)


# ============================================================
# Ablation vs baseline
# ============================================================

base_mae = float(overall.loc[overall["Variant"] == "Baseline", "MAE"].iloc[0])
base_rmse = float(overall.loc[overall["Variant"] == "Baseline", "RMSE"].iloc[0])

ablation = overall[overall["Variant"] != "Baseline"].copy()

ablation["Delta_MAE_V"] = base_mae - ablation["MAE"]
ablation["Delta_RMSE_V"] = base_rmse - ablation["RMSE"]
ablation["Delta_MAE_mV"] = 1000.0 * ablation["Delta_MAE_V"]
ablation["Delta_RMSE_mV"] = 1000.0 * ablation["Delta_RMSE_V"]

ablation["MAE_reduction_pct"] = (
    100.0 * ablation["Delta_MAE_V"] / base_mae
    if base_mae > 0 else np.nan
)

ablation["RMSE_reduction_pct"] = (
    100.0 * ablation["Delta_RMSE_V"] / base_rmse
    if base_rmse > 0 else np.nan
)

ablation_path = os.path.join(OUT_DIR, "sec7_ablation_vs_baseline.csv")
ablation.to_csv(ablation_path, index=False)


# ============================================================
# Band-wise Delta MAE
# ============================================================

band_rows = []

for band in SOC_LABELS:
    group = df[df["SoC_band"] == band].copy()

    if group.empty:
        continue

    base_band_mae = mae_from_residual(group["Res_Baseline"])

    band_rows.append({
        "SoC_band": band,
        "DeltaMAE_Thermal_mV": 1000.0 * (
            base_band_mae - mae_from_residual(group["Res_Thermal"])
        ),
        "DeltaMAE_TplusRate_mV": 1000.0 * (
            base_band_mae - mae_from_residual(group["Res_TR"])
        ),
        "N": int(len(group)),
    })

bandwise_delta = pd.DataFrame(band_rows)

bandwise_delta_path = os.path.join(OUT_DIR, "sec7_bandwise_delta_mae_mV.csv")
bandwise_delta.to_csv(bandwise_delta_path, index=False)


# ============================================================
# Parity regression
# ============================================================

parity_rows = []

for variant, pred_col in [
    ("Baseline", "Pred_Baseline"),
    ("Thermal", "Pred_Thermal"),
    ("T+Rate", "Pred_TR"),
]:
    tmp = df[[ycol, pred_col]].dropna().copy()

    true_v = tmp[ycol].to_numpy(dtype=float)
    pred_v = tmp[pred_col].to_numpy(dtype=float)

    if len(tmp) >= 2:
        slope, intercept = np.polyfit(true_v, pred_v, 1)
    else:
        slope, intercept = np.nan, np.nan

    parity_rows.append({
        "Variant": variant,
        "Slope": float(slope),
        "Intercept": float(intercept),
        "N": int(len(tmp)),
    })

parity_regression = pd.DataFrame(parity_rows)

parity_regression_path = os.path.join(OUT_DIR, "sec7_parity_regression.csv")
parity_regression.to_csv(parity_regression_path, index=False)


# ============================================================
# Save updated pred table with residuals
# ============================================================

pred_with_residuals_path = os.path.join(OUT_DIR, "pred_with_residuals.csv")
df.to_csv(pred_with_residuals_path, index=False)


# ============================================================
# Print tables
# ============================================================

print("\n-- Overall metrics --")
display(overall)

print("\n-- Ablation vs baseline --")
display(
    ablation[
        [
            "Variant",
            "Delta_MAE_mV",
            "Delta_RMSE_mV",
            "MAE_reduction_pct",
            "RMSE_reduction_pct",
        ]
    ].round(3)
)

print("\n-- Band-wise Delta MAE (mV) --")
display(bandwise_delta.round(3))

print("\n-- Parity regression: Predicted V = slope * True V + intercept --")
display(parity_regression.round(4))

print("\nSaved tables:")
print(" -", overall_path)
print(" -", ablation_path)
print(" -", bandwise_delta_path)
print(" -", parity_regression_path)
print(" -", pred_with_residuals_path)


# ============================================================
# Figure 1: Bias(SoC), mean residual per rounded 1%-SoC bin
# ============================================================

def bin_mean_residual(residual, soc):
    data = pd.DataFrame({
        "residual": residual,
        "SoC": soc,
    }).dropna()

    data["soc_bin"] = data["SoC"].round().clip(0, 100)

    return (
        data.groupby("soc_bin", as_index=False)["residual"]
        .mean()
        .sort_values("soc_bin")
    )


bias_baseline = bin_mean_residual(df["Res_Baseline"], df["SoC"])
bias_thermal = bin_mean_residual(df["Res_Thermal"], df["SoC"])
bias_tr = bin_mean_residual(df["Res_TR"], df["SoC"])

plt.figure(figsize=(6.2, 4.0))
plt.plot(bias_baseline["soc_bin"], bias_baseline["residual"], label="Baseline")
plt.plot(bias_thermal["soc_bin"], bias_thermal["residual"], label="Thermal")
plt.plot(bias_tr["soc_bin"], bias_tr["residual"], label="T+Rate")
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("SoC (%)")
plt.ylabel("Mean residual V (true - predicted)")
plt.title("Bias(SoC): mean residual per SoC bin")
plt.legend()
plt.tight_layout()

fig_bias_path = save_current_figure("fig_bias_mean_residual.png")
plt.show()
plt.close()


# ============================================================
# Figure 2: Parity triptych, true V vs predicted V
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(12.0, 4.0), sharex=True, sharey=True)

all_values = np.r_[
    df[ycol].to_numpy(dtype=float),
    df["Pred_Baseline"].to_numpy(dtype=float),
    df["Pred_Thermal"].to_numpy(dtype=float),
    df["Pred_TR"].to_numpy(dtype=float),
]

lo, hi = np.nanpercentile(all_values, [1, 99])

for ax, (name, pred_col) in zip(
    axes,
    [
        ("Baseline", "Pred_Baseline"),
        ("Thermal", "Pred_Thermal"),
        ("T+Rate", "Pred_TR"),
    ],
):
    ax.scatter(df[ycol], df[pred_col], s=6, alpha=0.25)
    ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1)
    ax.set_title(name)
    ax.set_xlabel("True V")
    ax.set_ylabel("Predicted V")

fig.suptitle("Parity: True vs Predicted")
plt.tight_layout()

fig_parity_path = save_current_figure("fig_parity_triptych.png")
plt.show()
plt.close()


# ============================================================
# Figure 3: Residual vs SoC cloud + running median
# ============================================================

def running_median_by_soc(soc, residual):
    data = pd.DataFrame({
        "SoC": soc,
        "residual": residual,
    }).dropna().sort_values("SoC")

    data["soc_bin"] = data["SoC"].round().clip(0, 100)

    return (
        data.groupby("soc_bin", as_index=False)["residual"]
        .median()
        .rename(columns={"soc_bin": "SoC", "residual": "median_residual"})
        .sort_values("SoC")
    )


fig, axes = plt.subplots(1, 3, figsize=(12.0, 4.0), sharey=True)

for ax, (name, res_col) in zip(
    axes,
    [
        ("Baseline", "Res_Baseline"),
        ("Thermal", "Res_Thermal"),
        ("T+Rate", "Res_TR"),
    ],
):
    ax.scatter(df["SoC"], df[res_col], s=4, alpha=0.15)

    med = running_median_by_soc(df["SoC"], df[res_col])
    ax.plot(med["SoC"], med["median_residual"], linewidth=2)

    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_title(name)
    ax.set_xlabel("SoC (%)")
    ax.set_ylabel("Residual (V)")

fig.suptitle("Residual vs SoC")
plt.tight_layout()

fig_residual_path = save_current_figure("fig_residual_vs_soc.png")
plt.show()
plt.close()


# ============================================================
# Run summary
# ============================================================

run_summary = {
    "input": {
        "pred": PRED_PATH,
        "true_voltage_column": ycol,
    },
    "outputs": {
        "overall_metrics_from_pred": overall_path,
        "sec7_ablation_vs_baseline": ablation_path,
        "sec7_bandwise_delta_mae_mV": bandwise_delta_path,
        "sec7_parity_regression": parity_regression_path,
        "pred_with_residuals": pred_with_residuals_path,
        "fig_bias_mean_residual": fig_bias_path,
        "fig_parity_triptych": fig_parity_path,
        "fig_residual_vs_soc": fig_residual_path,
    },
    "variants": {
        "Baseline": "anchor LUT at T=25, Rate=C/40, Charge_Rate=1C",
        "Thermal": "temperature-indexed LUT at row T, Rate=C/40, Charge_Rate=1C",
        "T+Rate": "full measured-grid LUT at row T, row Rate, row Charge_Rate",
    },
    "terminology": {
        "Rate": "charge cut-off current level",
        "Charge_Rate": "discharge-rate/rate-history level",
    },
    "n_rows_used": int(len(df)),
}

run_summary_path = os.path.join(OUT_DIR, "sec7_diagnostics_run_summary.json")

with open(run_summary_path, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2, ensure_ascii=False)

print("\nSaved figures:")
print(" -", fig_bias_path)
print(" -", fig_parity_path)
print(" -", fig_residual_path)

print("\nRun summary saved:")
print(" -", run_summary_path)

In [ ]:
# ============================================================
# Section 7 - Optional LUT memory report
# GitHub/reproducibility-ready single cell
#
# Notebook:
#   06_section7_lut_profile_evaluation.ipynb
#
# Reads:
#   outputs/06_section7_lut_profile_evaluation/lut_report.json
#   outputs/06_section7_lut_profile_evaluation/lut_4d.csv
#
# Output:
#   outputs/06_section7_lut_profile_evaluation/lut_memory_report.json
#
# Purpose:
#   Create a compact memory-footprint report for the 4D LUT and
#   per-temperature 3D LUTs.
# ============================================================

from pathlib import Path
import json
import pandas as pd

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

SECTION7_DIR = Path("outputs/06_section7_lut_profile_evaluation")
OUT_DIR = SECTION7_DIR

OUT_DIR.mkdir(parents=True, exist_ok=True)

lut_report_path = SECTION7_DIR / "lut_report.json"
lut4d_csv_path = SECTION7_DIR / "lut_4d.csv"

if not lut_report_path.exists() and not lut4d_csv_path.exists():
    raise FileNotFoundError(
        "Could not find LUT source files. Run the Section 7 isotonic "
        "LUT atlas cell first. Expected at least one of:\n"
        f" - {lut_report_path}\n"
        f" - {lut4d_csv_path}"
    )

print("LUT source directory:", SECTION7_DIR)

# ------------------------------------------------------------
# Infer LUT dimensions
# ------------------------------------------------------------

temperature_levels = []
rate_levels = []
charge_rate_levels = []
soc_points = 0

# First try lut_report.json.
if lut_report_path.exists():
    try:
        with open(lut_report_path, "r", encoding="utf-8") as f:
            lut_report = json.load(f)

        levels = lut_report.get("levels", {})
        soc_grid = lut_report.get("soc_grid", {})

        temperature_levels = (
            levels.get("Temperature", [])
            or lut_report.get("temps", [])
            or []
        )

        rate_levels = (
            levels.get("Rate_charge_cutoff", [])
            or lut_report.get("rates", [])
            or []
        )

        charge_rate_levels = (
            levels.get("Charge_Rate_ratehist", [])
            or lut_report.get("charge_levels", [])
            or lut_report.get("charges", [])
            or []
        )

        soc_points = int(
            soc_grid.get("n_points", 0)
            or lut_report.get("soc_points", 0)
            or 0
        )

        print("Dimension source: lut_report.json")

    except Exception as exc:
        print(f"Warning: lut_report.json could not be parsed: {exc}")

# Fallback to lut_4d.csv if needed.
if (
    not temperature_levels
    or not rate_levels
    or not charge_rate_levels
    or soc_points <= 0
):
    if not lut4d_csv_path.exists():
        raise FileNotFoundError(
            "lut_report.json did not contain complete dimensions, "
            "and lut_4d.csv was not found for fallback."
        )

    print("Dimension source: lut_4d.csv fallback")

    lut4d = pd.read_csv(lut4d_csv_path)

    required_cols = {"Temperature", "Rate", "Charge_Rate", "SoC"}
    missing_cols = required_cols - set(lut4d.columns)

    if missing_cols:
        raise KeyError(
            f"lut_4d.csv is missing required columns: {missing_cols}"
        )

    temperature_levels = sorted(pd.unique(lut4d["Temperature"]).tolist())
    rate_levels = sorted(pd.unique(lut4d["Rate"]).tolist())
    charge_rate_levels = sorted(pd.unique(lut4d["Charge_Rate"]).tolist())
    soc_points = int(lut4d["SoC"].nunique())

# ------------------------------------------------------------
# Compute memory footprint
# ------------------------------------------------------------

n_temperatures = len(temperature_levels)
n_rates = len(rate_levels)
n_charge_rates = len(charge_rate_levels)
n_soc = int(soc_points)

entries_4d = n_temperatures * n_rates * n_charge_rates * n_soc
entries_per_temperature_3d = n_rates * n_charge_rates * n_soc

def memory_bytes(n_entries, bytes_per_value):
    return int(n_entries * bytes_per_value)

memory_report = {
    "source_dir": str(SECTION7_DIR),
    "dimensions": {
        "Temperature_levels": temperature_levels,
        "Rate_charge_cutoff_levels": rate_levels,
        "Charge_Rate_ratehist_levels": charge_rate_levels,
        "n_temperatures": n_temperatures,
        "n_rates": n_rates,
        "n_charge_rates": n_charge_rates,
        "n_soc_points": n_soc,
    },
    "entries": {
        "entries_4d": entries_4d,
        "entries_per_temperature_3d": entries_per_temperature_3d,
        "expected_full_4x2x3x101_entries": 4 * 2 * 3 * 101,
    },
    "memory": {
        "float32_bytes": memory_bytes(entries_4d, 4),
        "float16_bytes": memory_bytes(entries_4d, 2),
        "per_temperature_float32_bytes": memory_bytes(entries_per_temperature_3d, 4),
        "per_temperature_float16_bytes": memory_bytes(entries_per_temperature_3d, 2),
    },
    "terminology": {
        "Rate": "charge cut-off current level",
        "Charge_Rate": "discharge-rate/rate-history level",
    },
}

memory_report["memory"]["float32_kB"] = round(
    memory_report["memory"]["float32_bytes"] / 1024.0,
    2,
)

memory_report["memory"]["float16_kB"] = round(
    memory_report["memory"]["float16_bytes"] / 1024.0,
    2,
)

memory_report["memory"]["per_temperature_float32_kB"] = round(
    memory_report["memory"]["per_temperature_float32_bytes"] / 1024.0,
    2,
)

memory_report["memory"]["per_temperature_float16_kB"] = round(
    memory_report["memory"]["per_temperature_float16_bytes"] / 1024.0,
    2,
)

# ------------------------------------------------------------
# Save report
# ------------------------------------------------------------

memory_report_path = OUT_DIR / "lut_memory_report.json"

with open(memory_report_path, "w", encoding="utf-8") as f:
    json.dump(memory_report, f, indent=2, ensure_ascii=False)

print("\n=== LUT memory report ===")
print(json.dumps({
    "source_dir": memory_report["source_dir"],
    "dimensions": memory_report["dimensions"],
    "entries": memory_report["entries"],
    "memory_kB": {
        "float32_kB": memory_report["memory"]["float32_kB"],
        "float16_kB": memory_report["memory"]["float16_kB"],
        "per_temperature_float32_kB": memory_report["memory"]["per_temperature_float32_kB"],
        "per_temperature_float16_kB": memory_report["memory"]["per_temperature_float16_kB"],
    },
}, indent=2, ensure_ascii=False))

print("\nSaved:")
print(memory_report_path)

if entries_4d == 0:
    print(
        "\nWARNING: entries_4d == 0. Check that lut_4d.csv or "
        "lut_report.json exists and contains valid LUT dimensions."
    )

In [ ]:
# ============================================================
# Section 7 - Bootstrap confidence intervals and Wilcoxon tests
# GitHub/reproducibility-ready single cell
#
# Notebook:
#   06_section7_lut_profile_evaluation.ipynb
#
# Required previous cell:
#   Section 7 measured-grid LUT prediction and evaluation
#
# Input:
#   outputs/06_section7_lut_profile_evaluation/pred.csv
#
# Uses paired absolute-error improvement:
#   Improvement = AbsErr_Baseline - AbsErr_variant
#
# Outputs:
#   outputs/06_section7_lut_profile_evaluation/sec7_bootstrap_wilcoxon.csv
#   outputs/06_section7_lut_profile_evaluation/sec7_bootstrap_wilcoxon_report.json
#
# Variants:
#   Thermal
#   T+Rate
#
# Interpretation note:
#   The bootstrap and Wilcoxon summaries are point-level paired diagnostics
#   over the measured-grid evaluation points. They should not be interpreted
#   as independent-cell inference.
# ============================================================

import os
import json
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# Paths
# ============================================================

OUT_DIR = "outputs/06_section7_lut_profile_evaluation"
os.makedirs(OUT_DIR, exist_ok=True)

PRED_PATH = os.path.join(OUT_DIR, "pred.csv")

if not os.path.exists(PRED_PATH):
    raise FileNotFoundError(
        "pred.csv was not found. Run the Section 7 measured-grid "
        "prediction cell first.\n"
        f"Missing file: {PRED_PATH}"
    )

print("Input:", PRED_PATH)


# ============================================================
# Load and validate prediction table
# ============================================================

df = pd.read_csv(PRED_PATH)

required_cols = {
    "Voltage_true",
    "Pred_Baseline",
    "Pred_Thermal",
    "Pred_TR",
}

missing_cols = required_cols - set(df.columns)

if missing_cols:
    raise KeyError(f"Missing required columns in pred.csv: {missing_cols}")

for col in required_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=list(required_cols)).copy()

if df.empty:
    raise RuntimeError("pred.csv has no usable rows after numeric filtering.")

# Recompute residuals and absolute errors defensively.
df["Res_Baseline"] = df["Voltage_true"] - df["Pred_Baseline"]
df["Res_Thermal"] = df["Voltage_true"] - df["Pred_Thermal"]
df["Res_TR"] = df["Voltage_true"] - df["Pred_TR"]

df["AbsErr_Baseline"] = df["Res_Baseline"].abs()
df["AbsErr_Thermal"] = df["Res_Thermal"].abs()
df["AbsErr_TR"] = df["Res_TR"].abs()

N = int(len(df))

print("Valid paired evaluation rows:", N)


# ============================================================
# Helper functions
# ============================================================

def rmse(residual):
    arr = np.asarray(residual, dtype=float)
    arr = arr[np.isfinite(arr)]

    if arr.size == 0:
        return np.nan

    return float(np.sqrt(np.mean(arr ** 2)))


def bootstrap_mean_ci(values, n_boot=1000, alpha=0.05, seed=42):
    """
    Row-level paired bootstrap CI for the mean improvement.
    """
    rng = np.random.default_rng(seed)

    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return np.nan, np.nan

    n = values.size
    boot_means = np.empty(n_boot, dtype=float)

    for i in range(n_boot):
        sample = rng.choice(values, size=n, replace=True)
        boot_means[i] = np.mean(sample)

    low, high = np.quantile(
        boot_means,
        [alpha / 2.0, 1.0 - alpha / 2.0],
    )

    return float(low), float(high)


def wilcoxon_paired_abs_errors(abs_base, abs_variant):
    """
    Paired Wilcoxon signed-rank test.

    H0:
        The median paired absolute-error improvement is zero.

    Alternative:
        Baseline absolute errors are greater than variant absolute errors.

    This uses paired differences:
        diff = AbsErr_Baseline - AbsErr_variant
    """
    try:
        from scipy.stats import wilcoxon

        x = np.asarray(abs_base, dtype=float)
        y = np.asarray(abs_variant, dtype=float)

        valid = np.isfinite(x) & np.isfinite(y)

        x = x[valid]
        y = y[valid]

        diff = x - y

        # Remove exact zeros for numerical stability and explicit reporting.
        diff_nonzero = diff[np.abs(diff) > 0]

        if diff_nonzero.size == 0:
            return np.nan, 0

        stat, p_value = wilcoxon(
            diff_nonzero,
            alternative="greater",
            zero_method="wilcox",
        )

        return float(p_value), int(diff_nonzero.size)

    except Exception as exc:
        print("[WARN] scipy Wilcoxon unavailable or failed:", repr(exc))
        return np.nan, 0


# ============================================================
# Baseline metrics
# ============================================================

base_mae = float(df["AbsErr_Baseline"].mean())
base_rmse = rmse(df["Res_Baseline"])

variant_specs = [
    {
        "Variant": "Thermal",
        "AbsErr": "AbsErr_Thermal",
        "Residual": "Res_Thermal",
    },
    {
        "Variant": "T+Rate",
        "AbsErr": "AbsErr_TR",
        "Residual": "Res_TR",
    },
]


# ============================================================
# Bootstrap CI and Wilcoxon tests
# ============================================================

N_BOOT = 1000
ALPHA = 0.05
SEED = 42

rows = []

for spec in variant_specs:
    variant = spec["Variant"]
    abs_col = spec["AbsErr"]
    res_col = spec["Residual"]

    variant_mae = float(df[abs_col].mean())
    variant_rmse = rmse(df[res_col])

    improvement_v = (
        df["AbsErr_Baseline"].to_numpy(dtype=float)
        - df[abs_col].to_numpy(dtype=float)
    )

    improvement_v = improvement_v[np.isfinite(improvement_v)]

    delta_mae_v = float(np.mean(improvement_v))

    ci_low_v, ci_high_v = bootstrap_mean_ci(
        improvement_v,
        n_boot=N_BOOT,
        alpha=ALPHA,
        seed=SEED,
    )

    p_wilcoxon, n_wilcoxon_nonzero = wilcoxon_paired_abs_errors(
        df["AbsErr_Baseline"].to_numpy(dtype=float),
        df[abs_col].to_numpy(dtype=float),
    )

    delta_rmse_v = base_rmse - variant_rmse

    rows.append({
        "Variant": variant,
        "N": N,
        "MAE": variant_mae,
        "RMSE": variant_rmse,

        "Delta_MAE_V": delta_mae_v,
        "Delta_MAE_mV": 1000.0 * delta_mae_v,

        "Delta_MAE_CI95_low_V": ci_low_v,
        "Delta_MAE_CI95_high_V": ci_high_v,
        "Delta_MAE_CI95_low_mV": 1000.0 * ci_low_v,
        "Delta_MAE_CI95_high_mV": 1000.0 * ci_high_v,

        "Delta_RMSE_V": delta_rmse_v,
        "Delta_RMSE_mV": 1000.0 * delta_rmse_v,

        "MAE_improvement_pct": (
            100.0 * delta_mae_v / base_mae
            if base_mae > 0 else np.nan
        ),
        "RMSE_improvement_pct": (
            100.0 * delta_rmse_v / base_rmse
            if base_rmse > 0 else np.nan
        ),

        "Wilcoxon_p_greater": p_wilcoxon,
        "Wilcoxon_N_nonzero": n_wilcoxon_nonzero,
    })

results = pd.DataFrame(rows)


# ============================================================
# Save outputs
# ============================================================

out_path = os.path.join(OUT_DIR, "sec7_bootstrap_wilcoxon.csv")
report_path = os.path.join(OUT_DIR, "sec7_bootstrap_wilcoxon_report.json")

results.to_csv(out_path, index=False)

report = {
    "input_pred": PRED_PATH,
    "n_rows": N,
    "baseline": {
        "MAE": base_mae,
        "RMSE": base_rmse,
    },
    "bootstrap": {
        "n_boot": N_BOOT,
        "alpha": ALPHA,
        "seed": SEED,
        "unit": "evaluation row",
        "quantity": (
            "paired mean improvement in absolute error: "
            "AbsErr_Baseline - AbsErr_variant"
        ),
    },
    "wilcoxon": {
        "alternative": "greater",
        "unit": "evaluation row after removing exact-zero paired differences",
        "interpretation": (
            "tests whether baseline absolute errors are larger than "
            "variant absolute errors"
        ),
    },
    "interpretation_note": (
        "These are point-level paired diagnostics over measured-grid "
        "evaluation rows. They should not be interpreted as independent-cell "
        "inference."
    ),
    "terminology": {
        "Rate": "charge cut-off current level",
        "Charge_Rate": "discharge-rate/rate-history level",
        "T+Rate": (
            "temperature, charge cut-off current, and "
            "discharge-rate/rate-history indexed LUT"
        ),
    },
    "outputs": {
        "csv": out_path,
        "json": report_path,
    },
}

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)


# ============================================================
# Print summary
# ============================================================

print("\n=== Section 7 Bootstrap CI + Wilcoxon tests ===")

display(
    results[
        [
            "Variant",
            "N",
            "Delta_MAE_mV",
            "Delta_MAE_CI95_low_mV",
            "Delta_MAE_CI95_high_mV",
            "Delta_RMSE_mV",
            "MAE_improvement_pct",
            "RMSE_improvement_pct",
            "Wilcoxon_p_greater",
            "Wilcoxon_N_nonzero",
        ]
    ].round(6)
)

print("\nSaved:")
print(" -", out_path)
print(" -", report_path)